# Biohub training kernel — train, predict, sweep, submit

End-to-end on one T4: warm-start from the official baseline weights, train with a
wall-clock budget (`--max-minutes`), run detection inference at a grid of
thresholds, solve the ILP linker on CPU for a grid of settings, and write one
`submission_*.csv` per combination. Each CSV can be submitted via
`competition_submit_code` against this kernel version — no extra GPU per submission.


## 1. Configuration


In [ ]:
import glob
import os

COMP_SLUG = "biohub-cell-tracking-during-development"

_comp_candidates = [
    p for p in glob.glob(f"/kaggle/input/*{COMP_SLUG}*", recursive=True)
    if os.path.isdir(os.path.join(p, "test"))
]
assert _comp_candidates, f"Competition input not mounted: {os.listdir('/kaggle/input')}"
COMP_DIR = sorted(_comp_candidates, key=len)[0]
TRAIN_DIR = f"{COMP_DIR}/train"
TEST_DIR = f"{COMP_DIR}/test"
print("Competition data:", COMP_DIR)

_candidates = [p[: -len("/wheels")] for p in glob.glob("/kaggle/input/**/wheels", recursive=True)]
assert _candidates, "Artifacts dataset not mounted"
ARTIFACTS = sorted(_candidates, key=len)[0]
print("Artifacts:", ARTIFACTS)

REPO_DIR = "/kaggle/working/repo"
METHOD = "unet_transformer"

# --- Training budget --------------------------------------------------------
# Wall-clock budget for training; the current epoch finishes first and the best
# checkpoint is always kept. Tune this so train + inference fit well under 12h.
MAX_MINUTES = 240          # calibration run; scale up once epoch time is known
EPOCHS = 200               # effectively unlimited; MAX_MINUTES is the real cap
LR = 1e-4
BATCH_SIZE = 16
# Warm-start the UNet from the official baseline checkpoint (3-epoch model).
WARM_START = True
# -----------------------------------------------------------------------------

# --- Inference grid ----------------------------------------------------------
DET_GRID = [0.99, 0.995]
DIV_GRID = [1.0]
# ------------------------------------------------------------------------------


## 2. GPU guard


In [ ]:
import torch

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    arch = torch.cuda.get_arch_list()
    print(f"GPU: {torch.cuda.get_device_name()} | capability {cap} | torch arches {arch}")
    assert cap >= (7, 0), f"GPU compute capability {cap} below minimum (sm_70). Aborting."
else:
    raise RuntimeError("No GPU available.")


## 3. Offline install & setup

Dependencies from the bundled wheels, then copy the repo and baseline weights.
The training script is overwritten with the current repo version (which supports
`--max-minutes`) in the next cell.


In [ ]:
import shutil
import subprocess

subprocess.run(
    ["pip", "install", "--no-index", "--find-links", f"{ARTIFACTS}/wheels",
     "tracksdata", "zarr>=3.0.10", "pyscipopt"],
    check=True,
)

shutil.copytree(f"{ARTIFACTS}/repo", REPO_DIR, dirs_exist_ok=True)
shutil.copytree(f"{ARTIFACTS}/weights", f"{REPO_DIR}/weights", dirs_exist_ok=True)


## 4. Patch in the current training script (`--max-minutes` support)


In [ ]:
train_src = '#!/usr/bin/env python\n"""\nTrain a temporal UNet + transformer edge predictor end-to-end.\n\nThe UNet (TemporalUNet3D) runs on each batch of consecutive frame pairs\nduring training.  Its output feature maps are indexed at integer node\ncoordinates (round + clamp), concatenated with sinusoidal positional\nembeddings, and fed to SimpleNodeTransformer.  Gradients flow back through\nthe integer indexing into the UNet weights.\n\nUsage:\n    uv run scripts/train_unet_transformer.py --split 0 --epochs 50\n"""\n\nimport argparse\nimport json\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport polars as pl\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport zarr\nfrom torch.utils.data import Dataset, DataLoader\nfrom tqdm import tqdm\n\nimport tracksdata as td\n\nfrom tracking_cellmot.io import invert_time_graph, open_dataset\nfrom tracking_cellmot.models import SimpleNodeTransformer, TemporalUNet3D\n\nfrom itertools import cycle as _cycle\n\n\ndef compute_gt_transition_matrix(\n    gt_ids_t: np.ndarray,\n    gt_ids_t1: np.ndarray,\n    edge_attrs: pl.DataFrame,\n) -> torch.Tensor:\n    """Build GT transition matrix directly as a float32 torch tensor."""\n    t_to_row = {nid: i for i, nid in enumerate(gt_ids_t)}\n    t1_to_col = {nid: i for i, nid in enumerate(gt_ids_t1)}\n\n    matrix = torch.zeros(len(gt_ids_t), len(gt_ids_t1), dtype=torch.float32)\n    for source_id, target_id in zip(edge_attrs["source_id"], edge_attrs["target_id"]):\n        if source_id in t_to_row and target_id in t1_to_col:\n            matrix[t_to_row[source_id], t1_to_col[target_id]] = 1.0\n\n    return matrix\n\n\ndef compute_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n    """BCE on annotated rows and columns (sparse GT — unannotated cells ignored)."""\n    active_rows = target.sum(dim=1) > 0\n    active_cols = target.sum(dim=0) > 0\n    mask = active_rows.unsqueeze(1) | active_cols.unsqueeze(0)\n    if not mask.any():\n        return torch.tensor(0.0, requires_grad=True, device=logits.device)\n\n    probs = torch.softmax(logits, dim=0)  # dim=0 intentional: divisions allowed, merges aren\'t\n    bce = F.binary_cross_entropy(probs, target, reduction="none")\n    p_t = probs * target + (1 - probs) * (1 - target)\n    loss = ((1 - p_t) ** 2) * bce\n\n    div_rows = target.sum(dim=1) > 1\n    weight = torch.ones_like(loss)\n    weight[div_rows] = 1.0\n\n    return (loss * weight)[mask].mean()\n\n\ndef compute_batch_loss(\n    logits: torch.Tensor,\n    target: torch.Tensor,\n    mask_t: torch.Tensor,\n    mask_t1: torch.Tensor,\n) -> torch.Tensor:\n    """Compute loss over a batch by slicing out real (unpadded) regions."""\n    B = logits.shape[0]\n    losses = []\n    for b in range(B):\n        nt = mask_t[b].sum().item()\n        nt1 = mask_t1[b].sum().item()\n        losses.append(compute_loss(logits[b, :nt, :nt1], target[b, :nt, :nt1]))\n    return torch.stack(losses).mean()\n\n\ndef _evaluate_pair(\n    logits: torch.Tensor,\n    target: torch.Tensor,\n) -> tuple[float, int, int]:\n    """Per-pair evaluation. Returns (loss, correct, total)."""\n    active_rows = target.sum(dim=1) > 0\n    active_cols = target.sum(dim=0) > 0\n    if not active_rows.any():\n        return 0.0, 0, 0\n\n    loss = compute_loss(logits, target).item()\n    probs = torch.softmax(logits, dim=0)\n    preds = (probs > 0.5).float()\n\n    mask = active_rows.unsqueeze(1) | active_cols.unsqueeze(0)\n    correct = (preds[mask] == target[mask]).sum().item()\n    total = mask.sum().item()\n\n    return loss, correct, total\n\nfrom augmentations import brightness_augment, flip_augment\n\nDEFAULT_AUGMENTATIONS = [brightness_augment, flip_augment]\nfrom dataspec import WEIGHTS_PATH\n\nDEFAULT_METHOD = "unet_transformer"\n_POS_EMBED_DIM = 8   # per axis; total = 4 axes × _POS_EMBED_DIM = 32\n\n\n# =============================================================================\n# Data structures\n# =============================================================================\n\n@dataclass(frozen=True)\nclass FrameWindowData:\n    """Metadata for a window of W consecutive frames (no image data stored).\n\n    ``t_start`` is the first frame index so the dataset can retrieve\n    ``image[t_start:t_start+n_frames]`` from zarr at batch time.\n    """\n\n    t_start: int                        # first frame index in the window\n    n_frames: int                       # W (window size)\n    pos_feats: list[torch.Tensor]       # W tensors, each (N_i, D)\n    coords: list[torch.Tensor]          # W tensors, each (N_i, 3)\n    node_counts: list[int]              # GT nodes per frame\n    targets: list[torch.Tensor]         # W-1 transition matrices\n\n\n@dataclass(frozen=True)\nclass VideoMeta:\n    """Lightweight per-video metadata enabling on-demand frame loading.\n\n    Workers open the zarr file themselves and load only the two frames\n    they need per batch item.  No full video tensor is kept in RAM\n    during training.\n    """\n\n    zarr_path: Path\n    image_shape: tuple[int, ...]  # (T, Z_ds, Y_ds, X_ds) after downsampling\n    downsample: tuple[int, ...]   # spatial downsample strides (Z, Y, X)\n    voxel_size: tuple[float, ...] # physical voxel size = scale * downsample\n    q_low: float                  # 0.1% quantile for normalization\n    q_high: float                 # 99.9% quantile for normalization\n\n\n# =============================================================================\n# Data preparation\n# =============================================================================\n\ndef extract_pos_features(\n    coords: np.ndarray,\n    image_shape: tuple[int, ...],\n    pos_embed_dim: int = _POS_EMBED_DIM,\n) -> np.ndarray:\n    """Sinusoidal positional embeddings for node coordinates (no intensity term).\n\n    Parameters\n    ----------\n    coords : np.ndarray\n        (N, 4) with columns [t, z, y, x].\n    image_shape : tuple\n        Full image shape (T, Z, Y, X) used for normalisation.\n    pos_embed_dim : int\n        Half-dimension per axis (sin half + cos half).\n\n    Returns\n    -------\n    np.ndarray\n        Shape (N, 4 * pos_embed_dim), float32.\n    """\n    t, z, y, x = coords[:, 0], coords[:, 1], coords[:, 2], coords[:, 3]\n    norms = [c / max(s, 1) for c, s in zip([t, z, y, x], image_shape)]\n\n    def _embed(vals: np.ndarray) -> np.ndarray:\n        freqs = 2 ** np.arange(pos_embed_dim // 2)\n        angles = vals[:, None] * freqs * np.pi\n        return np.concatenate([np.sin(angles), np.cos(angles)], axis=1)\n\n    return np.concatenate([_embed(n) for n in norms], axis=1).astype(np.float32)\n\n\ndef get_window_data(\n    gt_graph: td.graph.BaseGraph,\n    image_shape: tuple[int, ...],\n    t_start: int,\n    window_size: int = 2,\n    downsample: tuple[int, ...] = (1, 1, 1),\n) -> FrameWindowData | None:\n    """Build a FrameWindowData for ``window_size`` consecutive frames.\n\n    Coordinates are divided by *downsample* (Z, Y, X) so they match the\n    downsampled image grid.  *image_shape* should already be the\n    downsampled shape.\n\n    Returns ``None`` if any frame in the window has zero GT nodes.\n    """\n    gt_attrs = gt_graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])\n    edge_attrs = gt_graph.edge_attrs(attr_keys=["source_id", "target_id"])\n\n    ds = np.array(downsample, dtype=np.float32)  # (3,)\n\n    per_frame_ids: list[np.ndarray] = []\n    pos_feats: list[torch.Tensor] = []\n    coords_list: list[torch.Tensor] = []\n    node_counts: list[int] = []\n\n    for i in range(window_size):\n        t = t_start + i\n        gt_t = gt_attrs.filter(pl.col("t") == t)\n        if len(gt_t) == 0:\n            return None\n\n        gt_coords_t = gt_t.select(["z", "y", "x"]).to_numpy().astype(np.float32) / ds\n        gt_ids = gt_t["node_id"].to_numpy()\n        n_gt = len(gt_coords_t)\n\n        full_coords = np.column_stack([np.full(n_gt, t, dtype=np.float32), gt_coords_t])\n\n        pos_feats.append(torch.from_numpy(extract_pos_features(full_coords, image_shape)))\n        coords_list.append(torch.from_numpy(gt_coords_t))\n        per_frame_ids.append(gt_ids)\n        node_counts.append(n_gt)\n\n    targets: list[torch.Tensor] = []\n    for i in range(window_size - 1):\n        targets.append(compute_gt_transition_matrix(\n            per_frame_ids[i], per_frame_ids[i + 1], edge_attrs,\n        ))\n\n    return FrameWindowData(\n        t_start=t_start,\n        n_frames=window_size,\n        pos_feats=pos_feats,\n        coords=coords_list,\n        node_counts=node_counts,\n        targets=targets,\n    )\n\n\ndef pad_window(\n    window: FrameWindowData,\n    max_nodes: int,\n) -> dict[str, torch.Tensor]:\n    """Pad GT nodes to ``max_nodes``; returns metadata only.\n\n    Stores per-frame data as ``(W, max_nodes, ...)`` and per-pair targets as\n    ``(W-1, max_nodes, max_nodes)``.\n    """\n    W = window.n_frames\n    D = window.pos_feats[0].shape[1]\n    M = max_nodes\n\n    pos_feats = torch.zeros(W, M, D, dtype=torch.float32)\n    coords = torch.zeros(W, M, 3, dtype=torch.float32)\n    masks = torch.zeros(W, M, dtype=torch.bool)\n    node_counts = torch.zeros(W, dtype=torch.long)\n\n    for i in range(W):\n        n = window.node_counts[i]\n        pos_feats[i, :n] = window.pos_feats[i]\n        coords[i, :n] = window.coords[i]\n        masks[i, :n] = True\n        node_counts[i] = n\n\n    targets = torch.zeros(W - 1, M, M, dtype=torch.float32)\n    for i in range(W - 1):\n        nt = window.node_counts[i]\n        nt1 = window.node_counts[i + 1]\n        targets[i, :nt, :nt1] = window.targets[i]\n\n    return {\n        "t_start": window.t_start,\n        "n_frames": W,\n        "pos_feats": pos_feats,\n        "coords": coords,\n        "masks": masks,\n        "targets": targets,\n        "node_counts": node_counts,\n    }\n\n\nclass FrameWindowDataset(Dataset):\n    """Fixed-size dataset of UNet frame windows for batched training.\n\n    Accepts a list of ``(VideoMeta, windows)`` tuples — one per source video.\n    No image data is stored in RAM.  Each ``__getitem__`` call opens the zarr\n    file for that video, reads W frames, and normalises them on the CPU.\n    """\n\n    def __init__(\n        self,\n        video_data: list[tuple[VideoMeta, list[FrameWindowData]]],\n        max_nodes: int | None = None,\n        augmentations: list | None = None,\n    ):\n        all_windows = [w for _, windows in video_data for w in windows]\n        if max_nodes is None:\n            max_nodes = max(max(w.node_counts) for w in all_windows)\n\n        self.max_nodes = max_nodes\n        self.augmentations = augmentations or []\n\n        self._data: list[tuple[dict, VideoMeta]] = []\n        for video_meta, windows in video_data:\n            for window in windows:\n                meta = pad_window(window, max_nodes)\n                self._data.append((meta, video_meta))\n\n    def __len__(self) -> int:\n        return len(self._data)\n\n    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:\n        meta, vm = self._data[idx]\n        t_start = meta["t_start"]\n        W = meta["n_frames"]\n        dz, dy, dx = vm.downsample\n\n        z = zarr.open_group(str(vm.zarr_path), mode="r")["0"]\n        target_shape = list(vm.image_shape[1:])\n\n        # Strided read from zarr — spatial downsample at I/O time: (W, Z_ds, Y_ds, X_ds)\n        raw = z[t_start : t_start + W, ::dz, ::dy, ::dx].astype(np.float32)\n        imgs = torch.from_numpy((raw - vm.q_low) / (vm.q_high - vm.q_low + 1e-6)).clamp(0.0)\n\n        if list(imgs.shape[1:]) != target_shape:\n            imgs = F.interpolate(\n                imgs[:, None], size=target_shape,\n                mode="trilinear", align_corners=False,\n            )[:, 0]\n\n        if self.augmentations:\n            rng = np.random.default_rng()\n            c, m = meta["coords"], meta["masks"]\n            for aug in self.augmentations:\n                imgs, c, m = aug(imgs, c, m, rng=rng)\n            meta = {**meta, "coords": c, "masks": m}\n\n        return {\n            **meta,\n            "imgs": imgs.half(),  # (W, *spatial)\n            "image_shape": torch.tensor(vm.image_shape, dtype=torch.long),\n            "voxel_size": torch.tensor(vm.voxel_size, dtype=torch.float32),\n            "downsample": torch.tensor(vm.downsample, dtype=torch.float32),\n        }\n\n\ndef load_dataset_windows(\n    ds_path: Path,\n    window_size: int = 2,\n    invert_time: bool = False,\n    max_frames: int | None = None,\n    downsample: tuple[int, ...] = (1, 1, 1),\n) -> tuple[VideoMeta, list[FrameWindowData]]:\n    """Load per-window metadata and video stats for one dataset.\n\n    Uses ``open_dataset(load_image=False)`` to read zarr metadata and tracks\n    without loading the full image into RAM.\n\n    Returns\n    -------\n    tuple[VideoMeta, list[FrameWindowData]]\n        Lightweight video metadata and per-window node data.  No image tensor.\n    """\n    ds = open_dataset(ds_path, normalize=False, require_tracks=True,\n                      load_image=False, downsample=downsample)\n    if "0.001" not in ds.quantiles or "0.999" not in ds.quantiles:\n        raise ValueError(f"Zarr attrs missing image_statistics.quantiles for {ds_path}")\n\n    image_shape = ds.image_shape\n    tracks = ds.tracks\n    voxel_size = tuple(s * d for s, d in zip(ds.scale, downsample))\n\n    if invert_time:\n        tracks = invert_time_graph(tracks, max_t=image_shape[0])\n\n    if max_frames is not None:\n        image_shape = (max_frames, *image_shape[1:])\n        tracks = tracks.filter(td.NodeAttr("t") < max_frames).subgraph()\n\n    video_meta = VideoMeta(\n        zarr_path=ds.zarr_path,\n        image_shape=image_shape,\n        downsample=downsample,\n        voxel_size=voxel_size,\n        q_low=float(ds.quantiles["0.001"]),\n        q_high=float(ds.quantiles["0.999"]),\n    )\n\n    windows: list[FrameWindowData] = []\n    for t in range(image_shape[0] - window_size + 1):\n        data = get_window_data(tracks, image_shape, t, window_size, downsample=downsample)\n        if data is not None:\n            windows.append(data)\n\n    return video_meta, windows\n\n\n# =============================================================================\n# Model\n# =============================================================================\n\nclass UNetNodeTransformer(nn.Module):\n    """TemporalUNet3D encoder + SimpleNodeTransformer edge predictor.\n\n    Forward pass:\n      1. Stack frames t and t+1 → (B, 2, 1, *spatial) → UNet → (B, 2, C_feat, *spatial)\n      2. Integer-index feature maps at node coords (round + clamp; differentiable)\n      3. Concatenate with sinusoidal positional embeddings\n      4. Cross-attention transformer → (B, max_nodes, max_nodes) edge logits\n    """\n\n    def __init__(\n        self,\n        unet: nn.Module,\n        unet_out_channels: int,\n        pos_feat_dim: int,\n        hidden_dim: int = 128,\n        n_heads: int = 4,\n        n_blocks: int = 4,\n        dropout: float = 0.3,\n    ):\n        super().__init__()\n        self.unet = unet\n        self.unet_out_channels = unet_out_channels\n\n        self.detect_head = nn.Conv3d(unet_out_channels, 1, kernel_size=1)\n\n        self.transformer = SimpleNodeTransformer(\n            feat_dim=unet_out_channels + pos_feat_dim,\n            hidden_dim=hidden_dim,\n            n_heads=n_heads,\n            n_blocks=n_blocks,\n            dropout=dropout,\n        )\n\n    def _index_features(\n        self,\n        feat_maps: torch.Tensor,  # (B, C, *spatial)\n        coords: torch.Tensor,     # (B, max_nodes, 3)\n        mask: torch.Tensor,       # (B, max_nodes) bool\n    ) -> torch.Tensor:\n        """Integer-index feat_maps at node positions; padded slots → zeros.\n\n        Gradients flow through the *feature map values* but NOT through the\n        coordinates (integer indexing is non-differentiable w.r.t. position).\n        """\n        B, C = feat_maps.shape[:2]\n        spatial = feat_maps.shape[2:]\n        max_nodes = coords.shape[1]\n\n        out = torch.zeros(B, max_nodes, C, device=feat_maps.device, dtype=feat_maps.dtype)\n        for b in range(B):\n            nt = int(mask[b].sum().item())\n            if nt == 0:\n                continue\n            z = coords[b, :nt, 0].long().clamp(0, spatial[0] - 1)\n            y = coords[b, :nt, 1].long().clamp(0, spatial[1] - 1)\n            x = coords[b, :nt, 2].long().clamp(0, spatial[2] - 1)\n            out[b, :nt] = feat_maps[b, :, z, y, x].T\n        return out\n\n    def detect(\n        self,\n        frame: torch.Tensor,  # (*spatial) — single pre-downsampled frame\n    ) -> torch.Tensor:\n        """Run UNet + detection head on a single frame.\n\n        Returns\n        -------\n        torch.Tensor\n            (*spatial) detection logits at the input (already downsampled) resolution.\n        """\n        # Duplicate the frame into a fake pair so the temporal UNet can run.\n        pair = torch.stack([frame, frame], dim=0).unsqueeze(0).unsqueeze(2)  # (1, 2, 1, *spatial)\n        unet_out = self.unet(pair)          # (1, 2, C_feat, *spatial)\n        det = self.detect_head(unet_out[0, 0:1])  # (1, 1, *spatial)\n        return det[0, 0]  # (*spatial)\n\n    def encode(\n        self,\n        imgs: torch.Tensor,  # (B, W, *spatial) — already downsampled\n    ) -> tuple[torch.Tensor, list[torch.Tensor]]:\n        """Run UNet encoder on W pre-downsampled frames.\n\n        Returns ``(unet_out, det_logits)`` where *unet_out* is\n        ``(B, W, C_feat, *spatial)`` and *det_logits* is a list of W\n        tensors each ``(B, 1, *spatial)``.\n        """\n        window = imgs.unsqueeze(2)  # (B, W, 1, *spatial)\n        unet_out = self.unet(window)  # (B, W, C_feat, *spatial)\n        W = unet_out.shape[1]\n        det_logits = [self.detect_head(unet_out[:, i]) for i in range(W)]\n        return unet_out, det_logits\n\n    def predict_edges(\n        self,\n        unet_feat_src: torch.Tensor,  # (B, N_src, C_feat) pre-indexed\n        unet_feat_tgt: torch.Tensor,  # (B, N_tgt, C_feat) pre-indexed\n        coords_src: torch.Tensor,     # (B, N_src, 3)\n        coords_tgt: torch.Tensor,     # (B, N_tgt, 3)\n        pos_feat_src: torch.Tensor,   # (B, N_src, pos_feat_dim)\n        pos_feat_tgt: torch.Tensor,   # (B, N_tgt, pos_feat_dim)\n        mask_src: torch.Tensor,       # (B, N_src) bool\n        mask_tgt: torch.Tensor,       # (B, N_tgt) bool\n    ) -> torch.Tensor:\n        """Run transformer edge predictor on pre-indexed UNet features."""\n        feat_src = torch.cat([unet_feat_src, pos_feat_src], dim=-1)\n        feat_tgt = torch.cat([unet_feat_tgt, pos_feat_tgt], dim=-1)\n        return self.transformer(feat_src, feat_tgt, coords_src, coords_tgt, mask_src, mask_tgt)\n\n\n# =============================================================================\n# Detection loss\n# =============================================================================\n\n\ndef compute_detection_loss(\n    det_logits: torch.Tensor,\n    coords: torch.Tensor,\n    mask: torch.Tensor,\n    neg_weight: float = 0.1,\n) -> torch.Tensor:\n    """BCE detection loss: GT node voxels are positive, all others lightly penalised.\n\n    Positive and negative terms are normalised by count so that each\n    contributes unit magnitude before *neg_weight* scaling.\n\n    Parameters\n    ----------\n    det_logits : torch.Tensor\n        (B, 1, Z, Y, X) raw logits from the detection head.\n    coords : torch.Tensor\n        (B, max_nodes, 3) GT node coordinates in downsampled space.\n    mask : torch.Tensor\n        (B, max_nodes) boolean mask for real (non-padded) nodes.\n    neg_weight : float\n        Weight for negative (non-GT) voxels.  Positives get weight 1.0.\n    """\n    B = det_logits.shape[0]\n    spatial = det_logits.shape[2:]  # (Z, Y, X)\n    logits = det_logits[:, 0]  # (B, Z, Y, X)\n    target = torch.zeros_like(logits)\n\n    # Mark GT voxels as positive for each sample in the batch.\n    nt = mask.sum(dim=1).long()       # (B,)\n    for b in range(B):\n        n_gt = nt[b]\n        if n_gt <= 0:\n            continue\n        gt_coords = coords[b, :nt[b]]\n        zi = gt_coords[:, 0].long().clamp(0, spatial[0] - 1)\n        yi = gt_coords[:, 1].long().clamp(0, spatial[1] - 1)\n        xi = gt_coords[:, 2].long().clamp(0, spatial[2] - 1)\n        n_unique = len(torch.unique(torch.stack([zi, yi, xi], dim=1), dim=0))\n        if n_unique < n_gt:\n            import warnings\n            warnings.warn(\n                f"Sample {b}: {n_gt - n_unique}/{n_gt} GT nodes collapsed to "\n                f"duplicate voxels after downsampling — these are undetectable.",\n                stacklevel=2,\n            )\n        target[b, zi, yi, xi] = 1.0\n\n    # Per-sample normalisation: weight_pos = 1/n_pos, weight_neg = neg_weight/n_neg.\n    n_pos = target.reshape(B, -1).sum(dim=1).clamp(min=1)          # (B,)\n    n_neg = (target.numel() // B - n_pos).clamp(min=1)             # (B,)\n    # Broadcast to spatial dims.\n    shape = (B,) + (1,) * len(spatial)\n    w_pos = (1.0 / n_pos).reshape(shape)\n    w_neg = (neg_weight / n_neg).reshape(shape)\n    weight = torch.where(target == 1.0, w_pos, w_neg)\n\n    return F.binary_cross_entropy_with_logits(\n        logits, target, weight=weight, reduction="sum",\n    ) / B\n\n\n# =============================================================================\n# Detection → matching → edge targets (used during training, GPU-vectorised)\n# =============================================================================\n\n\ndef _pos_embed_torch(\n    coords: torch.Tensor,\n    image_shape: tuple[int, ...],\n    pos_embed_dim: int = _POS_EMBED_DIM,\n) -> torch.Tensor:\n    """Batched sinusoidal positional embeddings (pure torch, stays on device).\n\n    Parameters\n    ----------\n    coords : (*, 4) with columns [t, z, y, x].\n    image_shape : (T, Z, Y, X) for normalisation.\n\n    Returns\n    -------\n    torch.Tensor  shape (*, 4 * pos_embed_dim).\n    """\n    shape_t = torch.tensor(image_shape, dtype=torch.float32, device=coords.device)\n    norms = coords / shape_t.clamp(min=1)  # (*, 4)\n    freqs = (2.0 ** torch.arange(pos_embed_dim // 2, device=coords.device, dtype=torch.float32)) * torch.pi\n    parts = []\n    for ax in range(4):\n        angles = norms[..., ax].unsqueeze(-1) * freqs  # (*, D//2)\n        parts.extend([angles.sin(), angles.cos()])\n    return torch.cat(parts, dim=-1)\n\n\ndef detect_and_match(\n    det_logits: torch.Tensor,\n    gt_coords: torch.Tensor,\n    mask: torch.Tensor,\n    image_shape: tuple[int, ...],\n    det_threshold: float = 0.3,\n    pool_kernel_um: float = 5.0,\n    max_match_distance: float = 5.0,\n    voxel_size: tuple[float, ...] | None = None,\n    frame_index: int = 0,\n    window_size: int | None = None,\n) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:\n    """Detect peaks in logits, match to GT, return padded edge-prediction inputs.\n\n    All coordinates are in downsampled space.\n\n    Parameters\n    ----------\n    det_logits : (B, 1, Z, Y, X)\n        Raw detection logits.\n    gt_coords : (B, max_nodes, 3)\n        GT node coordinates (downsampled).\n    mask : (B, max_nodes)\n        Boolean mask for real (non-padded) nodes.\n    image_shape : (T, Z, Y, X) downsampled shape for pos-embed normalisation.\n    det_threshold : minimum logit to accept a peak.\n    pool_kernel_um : local-max suppression distance in microns.\n    max_match_distance : maximum physical distance for detection→GT matching.\n    voxel_size : (Z, Y, X) physical voxel size (scale * downsample).\n        When provided, distances are computed in physical units.\n\n    Returns\n    -------\n    coords : (B, M, 3)  detected coordinates (downsampled).\n    pos    : (B, M, 4*_POS_EMBED_DIM)  positional embeddings.\n    mask   : (B, M)  bool.\n    matches : list[Tensor]  per-sample match arrays.\n    """\n    B = det_logits.shape[0]\n    device = det_logits.device\n    vs = (\n        torch.tensor(voxel_size, dtype=torch.float32, device=device)\n        if voxel_size is not None\n        else None\n    )\n\n    # Convert physical suppression distance (microns) to per-axis voxel kernel.\n    if voxel_size is not None:\n        pool_kernel = tuple(\n            max(1, k if k % 2 == 1 else k + 1)\n            for k in (max(1, round(pool_kernel_um / s)) for s in voxel_size)\n        )\n    else:\n        k = max(1, round(pool_kernel_um))\n        pool_kernel = (k if k % 2 == 1 else k + 1,) * 3\n\n    pad = tuple(k // 2 for k in pool_kernel)\n\n    # --- 1. Batched local-max peak detection on GPU -------------------------\n    with torch.no_grad():\n        pooled = F.max_pool3d(det_logits, pool_kernel, stride=1, padding=pad)\n        is_peak = (det_logits == pooled) & (det_logits > det_threshold)\n        # (N_total, 4): columns [b, z, y, x]\n        peak_idx = torch.nonzero(is_peak[:, 0])\n\n    batch_ids = peak_idx[:, 0]                         # (N_total,)\n    peak_coords = peak_idx[:, 1:].float()              # (N_total, 3)\n\n    # --- 2. Per-sample matching (small loop, torch.cdist on GPU) ------------\n    nt_per_sample = mask.sum(dim=1).long()              # (B,)\n\n    sample_matches: list[torch.Tensor] = []             # each (n_det,) long\n    sample_coords: list[torch.Tensor] = []              # each (n_det, 3)\n    max_det = 0\n\n    for b in range(B):\n        sel = batch_ids == b\n        det_b = peak_coords[sel]                        # (n_det, 3)\n        n_det = det_b.shape[0]\n        nt = int(nt_per_sample[b].item())\n        gt_b = gt_coords[b, :nt]                        # (n_gt, 3)\n        n_gt = gt_b.shape[0]\n\n        matched = torch.full((n_det,), -1, dtype=torch.long, device=device)\n        if n_det > 0 and n_gt > 0:\n            if vs is not None:\n                dists = torch.cdist(det_b * vs, gt_b * vs)\n            else:\n                dists = torch.cdist(det_b, gt_b)\n            min_d, min_i = dists.min(dim=1)             # (n_det,)\n            order = min_d.argsort()\n            gt_taken = torch.zeros(n_gt, dtype=torch.bool, device=device)\n            for idx in order:\n                if min_d[idx] > max_match_distance:\n                    break\n                gi = min_i[idx]\n                if not gt_taken[gi]:\n                    matched[idx] = gi\n                    gt_taken[gi] = True\n\n        sample_matches.append(matched)\n        sample_coords.append(det_b)\n        if n_det > max_det:\n            max_det = n_det\n\n    max_det = max(max_det, 1)\n\n    # --- 3. Pad coords / pos / mask (on GPU) --------------------------------\n    padded_coords = torch.zeros(B, max_det, 3, device=device)\n    padded_mask = torch.zeros(B, max_det, dtype=torch.bool, device=device)\n    for b in range(B):\n        n = sample_coords[b].shape[0]\n        if n == 0:\n            continue\n        padded_coords[b, :n] = sample_coords[b]\n        padded_mask[b, :n] = True\n\n    # Positional embeddings (batched, on GPU).\n    # Use window-relative time (0, 1, ..., W-1) normalised by W, not absolute frame index.\n    t_col = torch.full((B, max_det, 1), frame_index, device=device, dtype=torch.float32)\n    full_coords = torch.cat([t_col, padded_coords], dim=-1)  # (B, M, 4)\n    pos_shape = (window_size,) + image_shape[1:] if window_size is not None else image_shape\n    padded_pos = _pos_embed_torch(full_coords, pos_shape)     # (B, M, D)\n\n    return padded_coords, padded_pos, padded_mask, sample_matches\n\n\ndef build_matched_edge_targets(\n    match_t: list[torch.Tensor],\n    match_t1: list[torch.Tensor],\n    gt_target: torch.Tensor,\n    max_det_t: int,\n    max_det_t1: int,\n) -> torch.Tensor:\n    """Build (B, max_det_t, max_det_t1) edge targets via vectorised indexing."""\n    B = gt_target.shape[0]\n    device = gt_target.device\n    target = torch.zeros(B, max_det_t, max_det_t1, device=device)\n\n    for b in range(B):\n        mt = match_t[b]                                 # (n_det_t,)\n        mt1 = match_t1[b]                               # (n_det_t1,)\n        gt_trans = gt_target[b]                          # (N_gt_t, N_gt_t1)\n        n_t, n_t1 = mt.shape[0], mt1.shape[0]\n        if n_t == 0 or n_t1 == 0:\n            continue\n\n        valid_t = mt >= 0\n        valid_t1 = mt1 >= 0\n        valid_mask = valid_t.unsqueeze(1) & valid_t1.unsqueeze(0)  # (n_t, n_t1)\n        safe_t = mt.clamp(min=0)\n        safe_t1 = mt1.clamp(min=0)\n        block = gt_trans[safe_t][:, safe_t1] * valid_mask.float()\n        target[b, :n_t, :n_t1] = block\n\n    return target\n\n\n# =============================================================================\n# Training\n# =============================================================================\n\ndef train_epoch(\n    model: UNetNodeTransformer,\n    loader: DataLoader,\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n    det_loss_weight: float = 0.1,\n    det_neg_weight: float = 0.1,\n    max_iters: int | None = None,\n    pool_kernel_um: float = 5.0,\n) -> tuple[float, float]:\n    """Train for one epoch, return (avg edge loss, avg detection loss).\n\n    When *max_iters* is set, the loader is cycled repeatedly until that many\n    iterations have been performed, regardless of dataset size.\n    """\n    model.train()\n    total_edge_loss = 0.0\n    total_det_loss = 0.0\n    n_samples = 0\n\n    if max_iters is not None:\n        batch_iter = _cycle(loader)\n        pbar = tqdm(range(max_iters), desc="  iters", leave=False, disable=False)\n    else:\n        batch_iter = iter(loader)\n        pbar = tqdm(range(len(loader)), desc="  batches", leave=False, disable=False)\n\n    t_data, t_forward, t_backward = 0.0, 0.0, 0.0\n    t0 = time.perf_counter()\n\n    for _ in pbar:\n        batch = next(batch_iter)\n\n        imgs = batch["imgs"].to(device, dtype=torch.float32, non_blocking=True)       # (B, W, *sp)\n        coords = batch["coords"].to(device, non_blocking=True)                         # (B, W, M, 3)\n        pos_feats = batch["pos_feats"].to(device, non_blocking=True)                   # (B, W, M, D)\n        masks = batch["masks"].to(device, non_blocking=True)                           # (B, W, M)\n        targets = batch["targets"].to(device, non_blocking=True)                       # (B, W-1, M, M)\n        image_shape = tuple(batch["image_shape"][0].tolist())\n        voxel_size = tuple(batch["voxel_size"][0].tolist())\n        ds_scale = batch["downsample"][0].to(device)                                   # (3,)\n\n        torch.cuda.synchronize()\n        t1 = time.perf_counter()\n        t_data += t1 - t0\n\n        B, W = imgs.shape[:2]\n\n        # --- 1. Encode: UNet features + detection logits --------------------\n        unet_out, det_logits = model.encode(imgs)\n        # unet_out: (B, W, C, *spatial),  det_logits: list of W × (B, 1, *spatial)\n\n        # --- 2. Detection loss over all W frames ---------------------------\n        det_losses = [\n            compute_detection_loss(\n                det_logits[i], coords[:, i], masks[:, i],\n                det_neg_weight,\n            )\n            for i in range(W)\n        ]\n        det_loss = sum(det_losses) / W\n\n        # --- 3. Per-frame detect → match → index UNet features -------------\n        frame_det: list[tuple[torch.Tensor, torch.Tensor, torch.Tensor,\n                              list[torch.Tensor], torch.Tensor]] = []\n        for i in range(W):\n            det_c, det_p, det_m, matches = detect_and_match(\n                det_logits[i], coords[:, i], masks[:, i],\n                image_shape,\n                voxel_size=voxel_size,\n                pool_kernel_um=pool_kernel_um,\n                frame_index=i, window_size=W,\n            )\n            unet_feat = model._index_features(\n                unet_out[:, i], det_c, det_m,\n            )\n            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n\n        # --- 4. Per-pair edge prediction and loss -------------------------\n        block_losses = []\n        for i in range(W - 1):\n            ns = frame_det[i][0].shape[1]\n            nt = frame_det[i + 1][0].shape[1]\n            pair_target = build_matched_edge_targets(\n                frame_det[i][3], frame_det[i + 1][3],\n                targets[:, i], ns, nt,\n            )\n            edge_logits = model.predict_edges(\n                frame_det[i][4], frame_det[i + 1][4],\n                frame_det[i][0] * ds_scale, frame_det[i + 1][0] * ds_scale,\n                frame_det[i][1], frame_det[i + 1][1],\n                frame_det[i][2], frame_det[i + 1][2],\n            )\n            block_losses.append(compute_batch_loss(\n                edge_logits, pair_target,\n                frame_det[i][2], frame_det[i + 1][2],\n            ))\n        edge_loss = sum(block_losses) / len(block_losses)\n\n        # --- 5. Combined loss -----------------------------------------------\n        loss = edge_loss + det_loss_weight * det_loss\n\n        torch.cuda.synchronize()\n        t2 = time.perf_counter()\n        t_forward += t2 - t1\n\n        optimizer.zero_grad()\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        optimizer.step()\n\n        torch.cuda.synchronize()\n        t3 = time.perf_counter()\n        t_backward += t3 - t2\n\n        total_edge_loss += edge_loss.item() * B\n        total_det_loss += det_loss.item() * B\n        n_samples += B\n\n        t0 = time.perf_counter()\n\n    t_total = t_data + t_forward + t_backward\n    if t_total > 0:\n        print(\n            f"  [timing] data: {t_data:.1f}s ({100*t_data/t_total:.0f}%) | "\n            f"forward: {t_forward:.1f}s ({100*t_forward/t_total:.0f}%) | "\n            f"backward: {t_backward:.1f}s ({100*t_backward/t_total:.0f}%) | "\n            f"total: {t_total:.1f}s"\n        )\n\n    return (\n        total_edge_loss / max(n_samples, 1),\n        total_det_loss / max(n_samples, 1),\n    )\n\n\n@torch.no_grad()\ndef evaluate(\n    model: UNetNodeTransformer,\n    loader: DataLoader,\n    device: torch.device,\n    pool_kernel_um: float = 5.0,\n) -> tuple[float, float, float]:\n    """Evaluate model using detect→match→predict (same path as training).\n\n    Returns (avg_loss, accuracy, node_recall).\n    """\n    model.eval()\n    total_loss, correct, total, n_pairs = 0.0, 0, 0, 0\n    gt_matched, gt_total = 0, 0\n\n    for batch in loader:\n        imgs = batch["imgs"].to(device, dtype=torch.float32, non_blocking=True)\n        coords = batch["coords"].to(device, non_blocking=True)\n        pos_feats = batch["pos_feats"].to(device, non_blocking=True)\n        masks = batch["masks"].to(device, non_blocking=True)\n        targets = batch["targets"].to(device, non_blocking=True)\n        image_shape = tuple(batch["image_shape"][0].tolist())\n        voxel_size = tuple(batch["voxel_size"][0].tolist())\n        ds_scale = batch["downsample"][0].to(device)\n\n        B, W = imgs.shape[:2]\n        unet_out, det_logits = model.encode(imgs)\n        frame_det: list[tuple[torch.Tensor, torch.Tensor, torch.Tensor,\n                              list[torch.Tensor], torch.Tensor]] = []\n        for i in range(W):\n            det_c, det_p, det_m, matches = detect_and_match(\n                det_logits[i], coords[:, i], masks[:, i],\n                image_shape,\n                voxel_size=voxel_size,\n                pool_kernel_um=pool_kernel_um,\n                frame_index=i, window_size=W,\n            )\n            unet_feat = model._index_features(\n                unet_out[:, i], det_c, det_m,\n            )\n            frame_det.append((det_c, det_p, det_m, matches, unet_feat))\n\n            # Node recall: how many GT nodes were matched by a detection?\n            for b in range(B):\n                n_gt = int(masks[b, i].sum().item())\n                n_matched = (matches[b] >= 0).sum().item()\n                gt_total += n_gt\n                gt_matched += n_matched\n\n        # Per-pair evaluation.\n        for i in range(W - 1):\n            ns = frame_det[i][0].shape[1]\n            nt = frame_det[i + 1][0].shape[1]\n            pair_target = build_matched_edge_targets(\n                frame_det[i][3], frame_det[i + 1][3],\n                targets[:, i], ns, nt,\n            )\n            pair_logits = model.predict_edges(\n                frame_det[i][4], frame_det[i + 1][4],\n                frame_det[i][0] * ds_scale, frame_det[i + 1][0] * ds_scale,\n                frame_det[i][1], frame_det[i + 1][1],\n                frame_det[i][2], frame_det[i + 1][2],\n            )\n\n            for b in range(B):\n                ns_b = int(frame_det[i][2][b].sum().item())\n                nt_b = int(frame_det[i + 1][2][b].sum().item())\n                pair_loss, pair_correct, pair_total = _evaluate_pair(\n                    pair_logits[b, :ns_b, :nt_b], pair_target[b, :ns_b, :nt_b],\n                )\n                total_loss += pair_loss\n                correct += pair_correct\n                total += pair_total\n                n_pairs += 1\n\n    node_recall = gt_matched / max(gt_total, 1)\n    return total_loss / max(n_pairs, 1), correct / max(total, 1), node_recall\n\n\n# =============================================================================\n# Main training function\n# =============================================================================\n\ndef train(\n    data_dir: Path,\n    fold: int,\n    splits_file: Path,\n    method: str = DEFAULT_METHOD,\n    n_epochs: int = 50,\n    lr: float = 1e-3,\n    batch_size: int = 16,\n    num_workers: int = 4,  # benchmark_preload.py: 4 workers, no pin_memory is optimal\n    unet_out_channels: int = 32,\n    unet_layers: list[int] | None = None,\n    unet_weights: Path | None = None,\n    downsample: tuple[int, ...] = (1, 4, 4),\n    det_loss_weight: float = 1e1,\n    det_neg_weight: float = 1e-2,\n    max_iters: int | None = None,\n    max_minutes: float | None = None,\n    debug_video: Path | None = None,\n    seed: int | None = None,\n    max_frames: int | None = None,\n    window_size: int = 2,\n    augmentations: list | None = DEFAULT_AUGMENTATIONS,\n    pool_kernel_um: float = 5.0,\n    data_parallel: bool = True,\n) -> UNetNodeTransformer:\n    """Train on one fold from a pre-computed splits file.\n\n    If *debug_video* is set the splits file is ignored and that single dataset\n    is used for both train and test (quick sanity-check / overfitting run).\n    """\n    if unet_layers is None:\n        unet_layers = [32, 64, 128]\n\n    if debug_video is not None:\n        train_files = test_files = [debug_video]\n        print(f"Debug mode: using single video {debug_video.name}", flush=True)\n    else:\n        if splits_file.exists():\n            folds = json.loads(splits_file.read_text())\n        else:\n            # No splits file: build a deterministic seed-0 split from the\n            # datasets in data_dir (90% train / 10% validation), matching the\n            # accompanying notebook. This makes --splits optional.\n            import random\n            stems = sorted(\n                p.name[:-5] for p in data_dir.glob("*.zarr")\n                if (data_dir / f"{p.name[:-5]}.geff").exists()\n            )\n            random.Random(0).shuffle(stems)\n            n_val = max(1, len(stems) // 10)\n            folds = [{"split": 0, "train": stems[n_val:], "test": stems[:n_val]}]\n            print(f"No splits file at {splits_file}; generated seed-0 split "\n                  f"({len(stems) - n_val} train / {n_val} val).", flush=True)\n        fold_data = folds[fold]\n        train_files = [data_dir / name for name in fold_data["train"]]\n        test_files = [data_dir / name for name in fold_data["test"]]\n        print(f"Fold {fold}: {len(train_files)} train, {len(test_files)} test", flush=True)\n\n    output_dir = WEIGHTS_PATH / method / f"split_{fold}"\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    # Save arch config so predict_unet_transformer can reconstruct the model.\n    model_config = {\n        "unet_out_channels": unet_out_channels,\n        "unet_layers": unet_layers,\n        "downsample": list(downsample),\n        "window_size": window_size,\n        "pool_kernel_um": pool_kernel_um,\n    }\n    (output_dir / "config.json").write_text(json.dumps(model_config, indent=2))\n\n    def _load(\n        files: list[Path], desc: str,\n    ) -> list[tuple[VideoMeta, list[FrameWindowData]]]:\n        print(f"Loading {desc} ({len(files)} datasets)...", flush=True)\n        data: list[tuple[VideoMeta, list[FrameWindowData]]] = []\n        for f in tqdm(files, desc=desc, disable=False):\n            video_meta, windows = load_dataset_windows(\n                f, window_size=window_size,\n                max_frames=max_frames,\n                downsample=downsample,\n            )\n            data.append((video_meta, windows))\n        n_windows = sum(len(w) for _, w in data)\n        print(f"  {desc} done: {n_windows} windows total", flush=True)\n        return data\n\n    train_video_data = _load(train_files, "train")\n    test_video_data = _load(test_files, "test")\n\n    # Compute consistent max_nodes across train + test.\n    all_windows = [w for _, ws in train_video_data + test_video_data for w in ws]\n    max_nodes = max(max(w.node_counts) for w in all_windows)\n    print(f"max_nodes={max_nodes}", flush=True)\n\n    pos_feat_dim = 4 * _POS_EMBED_DIM\n\n    train_ds = FrameWindowDataset(train_video_data, max_nodes=max_nodes, augmentations=augmentations)\n    test_ds = FrameWindowDataset(test_video_data, max_nodes=max_nodes)\n    g = None\n    worker_init_fn = None\n    if seed is not None:\n        g = torch.Generator()\n        g.manual_seed(seed)\n\n        def worker_init_fn(worker_id: int) -> None:\n            worker_seed = torch.initial_seed() % 2**32\n            np.random.seed(worker_seed)\n\n    train_loader = DataLoader(\n        train_ds, batch_size=batch_size, shuffle=True,\n        num_workers=num_workers, prefetch_factor=2 if num_workers > 0 else None,\n        persistent_workers=num_workers > 0, pin_memory=False,\n        generator=g, worker_init_fn=worker_init_fn,\n    )\n    test_loader = DataLoader(\n        test_ds, batch_size=batch_size, shuffle=False,\n        num_workers=num_workers, prefetch_factor=2 if num_workers > 0 else None,\n        persistent_workers=num_workers > 0, pin_memory=False,\n        generator=g, worker_init_fn=worker_init_fn,\n    )\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    n_visible = torch.cuda.device_count() if device.type == "cuda" else 0\n    print(f"Using device: {device} | visible CUDA GPUs: {n_visible}", flush=True)\n\n    unet = TemporalUNet3D(\n        in_channels=1,\n        out_channels=unet_out_channels,\n        layers=unet_layers,\n    )\n    if unet_weights is not None:\n        state = torch.load(unet_weights, map_location="cpu", weights_only=True)\n        missing, unexpected = unet.load_state_dict(state, strict=False)\n        print(f"  UNet weights: {len(missing)} missing, {len(unexpected)} unexpected", flush=True)\n\n    model = UNetNodeTransformer(\n        unet=unet,\n        unet_out_channels=unet_out_channels,\n        pos_feat_dim=pos_feat_dim,\n    ).to(device)\n\n    # Simple multi-GPU: split the heavy UNet pass across all visible GPUs.\n    # Only the UNet is wrapped (it takes/returns plain batched tensors); the\n    # detection head and transformer stay on cuda:0. Checkpoints are saved with\n    # the DataParallel "module." prefix stripped so they load on a single GPU.\n    if data_parallel and device.type == "cuda" and n_visible > 1:\n        model.unet = nn.DataParallel(model.unet)\n        print(\n            f"DataParallel: UNet split across {n_visible} GPUs "\n            f"(effective per-GPU batch {max(1, batch_size // n_visible)})",\n            flush=True,\n        )\n    elif device.type == "cuda":\n        reason = "--single-gpu set" if not data_parallel else f"only {n_visible} GPU visible"\n        print(f"Single-GPU training ({reason}). For 2 GPUs set the Kaggle accelerator to \'GPU T4 x2\'.", flush=True)\n\n    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    print(f"Model parameters: {n_params:,}", flush=True)\n\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)\n    print(f"Starting training for {n_epochs} epochs (batch_size={batch_size})...", flush=True)\n\n    best_score = 0.0\n    save_path = output_dir / "edge_predictor_best.pth"\n    pbar = tqdm(range(n_epochs), desc="Training", disable=False)\n    print(f"Detection loss: weight={det_loss_weight}, neg_weight={det_neg_weight}", flush=True)\n    t_train_start = time.monotonic()\n\n    for epoch in pbar:\n        t0 = time.monotonic()\n        edge_loss, det_loss = train_epoch(\n            model, train_loader, optimizer, device, det_loss_weight, det_neg_weight,\n            max_iters=max_iters, pool_kernel_um=pool_kernel_um,\n        )\n        train_time = time.monotonic() - t0\n\n        t0 = time.monotonic()\n        test_loss, test_acc, test_recall = evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um)\n        test_time = time.monotonic() - t0\n\n        score = test_acc * test_recall\n        is_best = score >= best_score\n\n        if is_best:\n            best_score = score\n            # Normalise any DataParallel "unet.module." prefix to "unet." so the\n            # checkpoint loads on a single GPU (e.g. in the prediction script).\n            torch.save(\n                {k.replace("unet.module.", "unet.", 1): v for k, v in model.state_dict().items()},\n                save_path,\n            )\n\n        marker = "*" if is_best else " "\n        pbar.set_postfix(edge=f"{edge_loss:.4f}", det=f"{det_loss:.4f}", acc=f"{test_acc:.4f}")\n        print(\n            f"  Epoch {epoch:3d}/{n_epochs} | edge={edge_loss:.4f} | det={det_loss:.4f} | "\n            f"test_loss={test_loss:.4f} | acc={test_acc:.4f} | recall={test_recall:.4f} | best={best_score:.4f} {marker} | "\n            f"train={train_time:.1f}s test={test_time:.1f}s",\n            flush=True,\n        )\n\n        elapsed_min = (time.monotonic() - t_train_start) / 60\n        if max_minutes is not None and elapsed_min >= max_minutes:\n            print(f"Time budget reached ({elapsed_min:.1f} min >= {max_minutes} min); "\n                  f"stopping after epoch {epoch}.", flush=True)\n            break\n\n    print(f"\\nBest score (acc*recall): {best_score:.4f}, saved to {save_path}", flush=True)\n    if save_path.exists():\n        state = torch.load(save_path, map_location=device, weights_only=True)\n        if isinstance(model.unet, nn.DataParallel):\n            state = {\n                (k.replace("unet.", "unet.module.", 1) if k.startswith("unet.") else k): v\n                for k, v in state.items()\n            }\n        model.load_state_dict(state)\n    return model\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(\n        description="Train UNet + transformer edge predictor end-to-end.",\n        formatter_class=argparse.RawDescriptionHelpFormatter,\n    )\n    parser.add_argument("--method", type=str, default=DEFAULT_METHOD)\n    parser.add_argument("--data-dir", type=str, default=None)\n    parser.add_argument("--splits", type=str, default=None)\n    parser.add_argument("--split", type=str, default="0",\n                        help="Split index (0-4) or \'all\'.")\n    parser.add_argument("--epochs", type=int, default=50)\n    parser.add_argument("--lr", type=float, default=1e-4)\n    parser.add_argument("--batch-size", type=int, default=16,\n                        help="Frames pairs per batch. All images in a fold must share "\n                             "the same spatial shape for batch_size > 1.")\n    parser.add_argument("--num-workers", type=int, default=8,\n                        help="DataLoader worker processes for parallel frame loading (default: 4).")\n    parser.add_argument("--unet-out-channels", type=int, default=32)\n    parser.add_argument("--unet-layers", type=str, default="32,64,128",\n                        help="Comma-separated UNet channel widths, shallow→deep.")\n    parser.add_argument("--unet-weights", type=str, default=None,\n                        help="Path to pretrained UNet weights; loaded with strict=False.")\n    parser.add_argument("--downsample", type=str, default="1,4,4",\n                        help="Comma-separated spatial downsample strides Z,Y,X (default: 1,4,4).")\n    parser.add_argument("--det-loss-weight", type=float, default=1e0,\n                        help="Weight for detection loss relative to edge loss (default: 1e1).")\n    parser.add_argument("--det-neg-weight", type=float, default=1e-2,\n                        help="Per-voxel weight for non-GT (negative) voxels in detection loss (default: 1e-2).")\n    parser.add_argument("--max-iters", type=int, default=None,\n                        help="Max training iterations per epoch. None = full epoch.")\n    parser.add_argument("--max-minutes", type=float, default=None,\n                        help="Stop training after this many minutes (finishes the "\n                             "current epoch first, keeping the best checkpoint).")\n    parser.add_argument("--debug-video", type=str, default=None,\n                        help="Path to a single dataset for quick debugging. "\n                             "Ignores --fold and splits file; trains and evaluates on this video only.")\n    parser.add_argument("--window-size", type=int, default=2,\n                        help="Number of consecutive frames per training window (default: 2).")\n    parser.add_argument("--pool-kernel-um", type=float, default=5.0,\n                        help="Local-max suppression distance in microns (default: 5.0).")\n    parser.add_argument("--data-parallel", dest="data_parallel", action="store_true", default=True,\n                        help="Split the UNet across all visible GPUs via nn.DataParallel "\n                             "when more than one is available (default: on).")\n    parser.add_argument("--single-gpu", dest="data_parallel", action="store_false",\n                        help="Disable multi-GPU; train on cuda:0 only.")\n\n    args = parser.parse_args()\n\n    from dataspec import DATASET_PATH\n    data_dir = Path(args.data_dir) if args.data_dir else Path(DATASET_PATH)\n    splits_file = Path(args.splits) if args.splits else data_dir / "dataset_splits.json"\n    unet_layers = [int(x) for x in args.unet_layers.split(",")]\n    unet_weights = Path(args.unet_weights) if args.unet_weights else None\n    debug_video = Path(args.debug_video) if args.debug_video else None\n    downsample = tuple(int(x) for x in args.downsample.split(","))\n\n    folds = [0] if debug_video is not None else (\n        range(5) if args.split == "all" else [int(args.split)]\n    )\n    for fold in folds:\n        train(\n            data_dir=data_dir,\n            fold=fold,\n            splits_file=splits_file,\n            method=args.method,\n            n_epochs=args.epochs,\n            lr=args.lr,\n            batch_size=args.batch_size,\n            num_workers=args.num_workers,\n            unet_out_channels=args.unet_out_channels,\n            unet_layers=unet_layers,\n            unet_weights=unet_weights,\n            downsample=downsample,\n            det_loss_weight=args.det_loss_weight,\n            det_neg_weight=args.det_neg_weight,\n            max_iters=args.max_iters,\n            max_minutes=args.max_minutes,\n            debug_video=debug_video,\n            window_size=args.window_size,\n            pool_kernel_um=args.pool_kernel_um,\n            data_parallel=args.data_parallel,\n        )\n\n\nif __name__ == "__main__":\n    main()\n'

with open(f"{REPO_DIR}/scripts/train_unet_transformer.py", "w") as f:
    f.write(train_src)
print("train_unet_transformer.py patched:", len(train_src), "chars")


## 5. Embryo-disjoint splits

Samples sharing the prefix before `_` are the same embryo and must stay in the
same fold. Hold out one embryo group for validation (model selection); train on
the rest. The notebook computes groups at runtime since the train mount is not
visible locally.


In [ ]:
import json

train_stems = sorted(f[:-5] for f in os.listdir(TRAIN_DIR) if f.endswith(".zarr"))
groups = {}
for s in train_stems:
    groups.setdefault(s.split("_")[0], []).append(s)
print(f"{len(train_stems)} train videos in {len(groups)} embryo groups: "
      + ", ".join(f"{g}({len(v)})" for g, v in sorted(groups.items())))

# Hold out the smallest embryo group as validation.
val_group = min(groups, key=lambda g: len(groups[g]))
folds = [{"split": 0,
          "train": [s for g, v in groups.items() if g != val_group for s in v],
          "test": groups[val_group]}]
with open(f"{REPO_DIR}/kaggle_train_splits.json", "w") as f:
    json.dump(folds, f)
print(f"val group: {val_group} ({len(groups[val_group])} videos) | "
      f"train: {len(folds[0]['train'])} videos")


## 6. Train (time-boxed)


In [ ]:
import subprocess

cmd = [
    "python", "scripts/train_unet_transformer.py",
    "--data-dir", TRAIN_DIR, "--splits", "kaggle_train_splits.json", "--split", "0",
    "--method", METHOD, "--epochs", str(EPOCHS), "--lr", str(LR),
    "--batch-size", str(BATCH_SIZE), "--max-minutes", str(MAX_MINUTES),
]
if WARM_START:
    cmd += ["--unet-weights", f"{REPO_DIR}/weights/{METHOD}/split_0/edge_predictor_best.pth"]
print(" ".join(cmd), flush=True)
subprocess.run(cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": "src"}, check=True)


## 7. Inference — candidate graphs at each detection threshold


In [ ]:
import importlib.util
from pathlib import Path

import torch

spec = importlib.util.spec_from_file_location(
    "put", f"{REPO_DIR}/scripts/predict_unet_transformer.py"
)
put = importlib.util.module_from_spec(spec)
spec.loader.exec_module(put)

from tracking_cellmot.io import save_graph

test_stems = sorted(f[:-5] for f in os.listdir(TEST_DIR) if f.endswith(".zarr"))

device = torch.device("cuda")
weights = Path(REPO_DIR) / "weights" / METHOD / "split_0" / "edge_predictor_best.pth"
model, window_size, downsample = put.load_model(weights, device)

candidate_geffs = {}
for det_thr in DET_GRID:
    cfg = put.PredictConfig(det_threshold=det_thr, use_ilp=True)
    out_dir = Path(REPO_DIR) / "candidates" / f"det_{det_thr:.3f}"
    out_dir.mkdir(parents=True, exist_ok=True)
    candidate_geffs[det_thr] = {}
    for name in test_stems:
        coords, edges = put.predict_video(
            model, Path(TEST_DIR) / name, device, cfg=cfg,
            window_size=window_size, downsample=downsample,
        )
        graph = put.build_graph(coords, edges)
        geff_path = out_dir / f"{name}.geff"
        save_graph(graph, geff_path)
        candidate_geffs[det_thr][name] = geff_path
        print(f"det={det_thr} {name}: {graph.num_nodes()} nodes, {graph.num_edges()} edges", flush=True)
print("candidate graphs done")


## 8. ILP sweep (CPU) — one CSV per (det, division) setting


In [ ]:
import pandas as pd
import tracksdata as td


def write_csv(graphs, csv_path):
    rows = []
    for name, graph in graphs.items():
        for r in graph.node_attrs().iter_rows(named=True):
            rows.append({
                "dataset": name, "row_type": "node", "node_id": int(r["node_id"]),
                "t": int(r["t"]), "z": int(round(r["z"])), "y": int(round(r["y"])),
                "x": int(round(r["x"])), "source_id": -1, "target_id": -1,
            })
        for r in graph.edge_attrs().iter_rows(named=True):
            rows.append({
                "dataset": name, "row_type": "edge", "node_id": -1,
                "t": -1, "z": -1, "y": -1, "x": -1,
                "source_id": int(r["source_id"]), "target_id": int(r["target_id"]),
            })
    df = pd.DataFrame(rows)
    df.insert(0, "id", range(len(df)))
    df.to_csv(csv_path, index=False)
    print(f"Wrote {csv_path} with {len(df)} rows", flush=True)


for det_thr, geffs in candidate_geffs.items():
    graphs = {}
    for name, gpath in geffs.items():
        graph = td.graph.IndexedRXGraph.from_geff(gpath)
        graphs[name] = graph[0] if isinstance(graph, tuple) else graph
    for div_w in DIV_GRID:
        solved = {}
        for name, graph in graphs.items():
            if graph.num_edges() == 0:
                solved[name] = graph
                continue
            solver = td.solvers.ILPSolver(
                edge_weight=-1.0 * td.EdgeAttr("edge_prob"),
                appearance_weight=0.1,
                disappearance_weight=0.1,
                division_weight=div_w,
            )
            with put.suppress_output():
                solved[name] = solver.solve(graph)
        tag = f"det{det_thr:.3f}_div{div_w:.1f}"
        out = Path(f"/kaggle/working/configs/{tag}")
        out.mkdir(parents=True, exist_ok=True)
        write_csv(solved, str(out / "submission.csv"))
        print(f"config -> configs/{tag}/submission.csv", flush=True)

# Top-level fallback (most promising config), per the competition filename rule.
import shutil
shutil.copyfile("/kaggle/working/configs/det0.990_div1.0/submission.csv",
                "/kaggle/working/submission.csv")
print("top-level submission.csv <- det0.990_div0.2")


## Submissions

Submit each CSV via `competition_submit_code(kernel="aakashkavuru/biohub-train-v1",
kernel_version=N, file_name="submission_detX_divY.csv")` — zero extra GPU.
